# Phân loại Parkinson từ giọng nói — đánh giá không rò rỉ theo bệnh nhân

Notebook này là bản trình bày gọn của mã trong `src/`. Đơn vị chia dữ liệu là **bệnh nhân**, không phải bản ghi. Mục tiêu là đánh giá đáng tin cậy, không chạy theo một con số Accuracy cao.

> Chỉ phục vụ nghiên cứu và học tập, không dùng để chẩn đoán.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data import SUBJECT_COLUMN, TARGET_COLUMN, build_subject_table, load_data, subject_holdout_split
from src.evaluate import calculate_metrics, make_subject_folds, positive_score
from src.features import MODEL_FEATURES
from src.train import RANDOM_STATE, train

df = load_data(PROJECT_ROOT / 'data' / 'parkinsons.csv')
print(f'{len(df)} bản ghi, {df[SUBJECT_COLUMN].nunique()} bệnh nhân, 22 đặc trưng gốc')

195 bản ghi, 32 bệnh nhân, 22 đặc trưng gốc


## Vì sao phép chia ngẫu nhiên theo dòng sai?

Mỗi người có nhiều bản ghi giọng nói. Nếu chia từng dòng, mô hình có thể gặp giọng của cùng một người ở cả train và test. Kết quả khi đó đo khả năng nhận ra người đã thấy nhiều hơn là khả năng khái quát sang bệnh nhân mới.

In [2]:
from sklearn.model_selection import train_test_split
naive_train, naive_test = train_test_split(df.index, test_size=0.2, random_state=2, stratify=df[TARGET_COLUMN])
overlap = set(df.loc[naive_train, SUBJECT_COLUMN]) & set(df.loc[naive_test, SUBJECT_COLUMN])
print(f'Chia theo dòng: {len(overlap)}/{df.loc[naive_test, SUBJECT_COLUMN].nunique()} bệnh nhân test đã xuất hiện trong train')

Chia theo dòng: 27/27 bệnh nhân test đã xuất hiện trong train


## Holdout và cross-validation đúng đơn vị

Ta tạo bảng một dòng cho mỗi `subject_id`, stratify trên bảng này, rồi ánh xạ danh sách bệnh nhân về các bản ghi. Mỗi validation fold được kiểm tra bắt buộc có cả hai lớp.

In [3]:
train_df, test_df = subject_holdout_split(df, random_state=RANDOM_STATE)
folds = make_subject_folds(train_df, n_splits=5, random_state=RANDOM_STATE)
audit = []
for fold, (fit_idx, valid_idx) in enumerate(folds, 1):
    fit, valid = train_df.iloc[fit_idx], train_df.iloc[valid_idx]
    assert set(fit[SUBJECT_COLUMN]).isdisjoint(valid[SUBJECT_COLUMN])
    assert valid[TARGET_COLUMN].nunique() == 2
    audit.append({'fold': fold, 'valid_subjects': valid[SUBJECT_COLUMN].nunique(), 'class_0': (valid[TARGET_COLUMN] == 0).sum(), 'class_1': (valid[TARGET_COLUMN] == 1).sum()})
pd.DataFrame(audit)

,fold,valid_subjects,class_0,class_1
0,1,5,6,24
1,2,5,6,24
2,3,5,6,25
3,4,5,12,18
4,5,4,6,19


## Benchmark sáu mô hình

Scaler và `SelectKBest` nằm trong pipeline. SVM dùng `decision_function` cho ROC–AUC; không bật `probability=True`, vì bước calibration nội bộ đó không nhận biết nhóm bệnh nhân.

In [4]:
benchmark = train(PROJECT_ROOT / 'data' / 'parkinsons.csv', PROJECT_ROOT / 'artifacts')
assert benchmark['CV ROC-AUC'].notna().all()
benchmark.round(4)

,Model,CV F1-macro mean,CV F1-macro std,CV Balanced Accuracy,CV ROC-AUC,Best parameters
3,"SVM (RBF, decision score)",0.7071,0.2524,0.7333,0.6725,"{""model__C"": 1, ""model__class_weight"": null, ""..."
2,KNN,0.6974,0.1772,0.7253,0.7272,"{""model__n_neighbors"": 3, ""model__p"": 2, ""mode..."
4,Random Forest,0.6582,0.2119,0.7087,0.6424,"{""model__class_weight"": null, ""model__max_dept..."
5,HistGradientBoosting,0.6521,0.2064,0.6991,0.6754,"{""model__l2_regularization"": 0, ""model__learni..."
1,Logistic Regression,0.6198,0.1885,0.6835,0.6953,"{""model__C"": 0.1, ""model__class_weight"": null,..."
0,Dummy,0.4284,0.0272,0.5000,0.5000,{}


## Độ ổn định

Repeated CV tiếp tục chia trên bảng bệnh nhân. Số split là 3 vì tập train chỉ có 6 bệnh nhân lớp 0; mỗi fold vẫn được assert có đủ hai lớp. Các fold không hợp lệ không bị âm thầm đưa vào giá trị trung bình.

In [5]:
import joblib
bundle = joblib.load(PROJECT_ROOT / 'artifacts' / 'parkinsons_champion_pipeline.joblib')
stability = []
for repeat in range(10):
    for fold, (fit_idx, valid_idx) in enumerate(make_subject_folds(train_df, n_splits=3, random_state=RANDOM_STATE + 100 + repeat), 1):
        model = clone(bundle['model']).fit(train_df.iloc[fit_idx][MODEL_FEATURES], train_df.iloc[fit_idx][TARGET_COLUMN])
        valid = train_df.iloc[valid_idx]
        score = positive_score(model, valid[MODEL_FEATURES])
        pred = (score >= 0.5).astype(int)
        fold_records = pd.DataFrame({'subject_id': valid[SUBJECT_COLUMN].to_numpy(), 'actual': valid[TARGET_COLUMN].to_numpy(), 'score': score, 'pred': pred})
        subjects = fold_records.groupby('subject_id', as_index=False).agg(actual=('actual', 'first'), score=('score', 'mean'))
        assert subjects['actual'].nunique() == 2
        subjects['pred'] = (subjects['score'] >= 0.5).astype(int)
        stability.append({'repeat': repeat + 1, 'fold': fold, **calculate_metrics(subjects['actual'], subjects['pred'], subjects['score'])})
stability_df = pd.DataFrame(stability)
stability_df[['F1-macro', 'Balanced Accuracy', 'ROC-AUC']].agg(['mean', 'std', 'min', 'max']).round(4)

,F1-macro,Balanced Accuracy,ROC-AUC
mean,0.6739,0.6694,0.8444
std,0.1852,0.1375,0.1410
min,0.3846,0.4167,0.4167
max,1.0000,1.0000,1.0000


## Kết luận

Accuracy 97,44% của cách chia theo dòng không phải bằng chứng tốt về khả năng khái quát: 27/27 bệnh nhân test cũng có bản ghi trong train. Sau khi đổi đơn vị đánh giá sang bệnh nhân, độ biến động giữa các fold lớn hơn và phản ánh đúng giới hạn của bộ dữ liệu nhỏ (32 người, chỉ 8 người lớp 0). Đây là kết quả trung thực hơn để trình bày trong portfolio.